[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_tuning.ipynb)

# Part A: tuning the classifiers

Two settings, tuned on val only, on 10% of the labels: **CNN-3 fine-tuning** and the **ResNet-18 linear probe**. The other classifiers (the CNN-3 linear probe, ResNet-18 fine-tuning, the supervised baselines) keep the settings in `config.py`. The DAE's own hyperparameters were tuned in search 1 and 2.

Three grids, one after the other. In each, **one hyperparameter at a time**, with 5 values, and the others fixed at what was chosen so far:

- **Grid 1, learning:** learning rate, the head's learning rate (fine-tuning only: in the linear probe only the head trains), batch size.
- **Grid 2, regularization:** dropout, augmentation, label smoothing.
- **Grid 3, optimizer:** the optimizer (AdamW, Adam, NAdam, RAdam, SGD with Nesterov momentum; SGD with a 10 times larger learning rate, its usual range), and its weight decay. Weight decay comes right after the optimizer because Adam and AdamW differ only in how they apply it. The encoders keep the activation they were pretrained with: another one would change the function their weights learned.

The best value of each, by val accuracy, is kept for the next one. Every run uses seed 42; the seeds come after the tuning, in `colab_final.ipynb`. Every run: up to 50 rounds (a round is as many images as 100% of the labels), stopped after 3 rounds with no better val accuracy (`--patience 3`; the final runs use 5). `--tune`: no test set. The choice of a setting goes to `midterm_runs/tuning_choice.json` when all its grids are done, so `colab_final.ipynb` never reads half a tuning.

On one L4 runtime: CNN-3 about 3 hours, ResNet-18 about 4.5 hours (its pretraining first).

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU.

In [ ]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

## Drive
Every run writes to `midterm_runs`, next to the earlier runs. The tuning runs have `tune` in their folder name.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_runs"
assert os.path.isdir(OUT), f"no folder {OUT}"

## TensorBoard
Filter the runs with `tune`, then *Toggle all runs*. The `epoch` section has train and val by round, the TEXT tab each run's `run_info`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## The search
`tune` runs the hyperparameters of one setting in the order given, 5 values each, and keeps the best. Runs that finished before are not trained again (train.py finds their folder), so after a disconnect, run the same cell again: every setting starts from the same values, so it repeats the same runs, and the finished ones are skipped.

In [ ]:
import importlib
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

VALUES = {
    "lr": [3e-5, 1e-4, 3e-4, 1e-3, 3e-3],
    "head_lr_mult": [1, 3, 10, 30, 100],
    "batch_size": [32, 64, 128, 256, 512],
    "wd": [0, 1e-5, 1e-4, 5e-4, 1e-3],
    "dropout": [0, 0.1, 0.25, 0.4, 0.5],
    "augment": ["none", "flip", "crop", "crop_rotate", "crop_jitter"],
    "label_smoothing": [0, 0.05, 0.1, 0.15, 0.2],
    "optimizer": ["adamw", "adam", "nadam", "radam", "sgd"],
}
GRID_1 = ["lr", "head_lr_mult", "batch_size"]  # learning; the head's lr: fine-tuning only
GRID_2 = ["dropout", "augment", "label_smoothing"]  # regularization
GRID_3 = ["optimizer", "wd"]  # optimizer
SGD_LR = 10  # SGD's learning rate, as a multiple of the Adam ones
DEFAULTS = {"head_lr_mult": 1, "batch_size": 256, "wd": 0, "dropout": 0, "augment": "flip", "label_smoothing": 0.1,
            "optimizer": "adamw"}
CHOICE_FILE = f"{OUT}/tuning_choice.json"
R = "--backbone resnet18 --decoder resnet18"  # the ResNet-18 DAE


def folder_of(flags):
    """The run folder train.py uses for these flags (run_info.folder_name)."""
    import config, utils, run_info
    for module in (config, utils, run_info):
        importlib.reload(module)
    sys.argv = ["train.py", *flags.split(), "--output_dir", OUT]
    return f"{OUT}/{run_info.folder_name(utils.parse_args().mode)}"


def train(flags):
    """One run of train.py, and its val numbers."""
    !python train.py {flags} --output_dir "{OUT}"
    with open(f"{folder_of(flags)}/results.json") as f:
        return json.load(f)


def tune(setting, base, lr, order):
    """Tune setting one hyperparameter at a time, from the defaults; save and return its choice."""
    choice = {"lr": lr, **DEFAULTS}
    for name in order:
        rows = []
        adam_lr = float(f"{choice['lr'] / SGD_LR:.3g}") if choice["optimizer"] == "sgd" else choice["lr"]
        for value in VALUES[name]:
            trial = {**choice, name: value}
            if name == "optimizer":  # each optimizer with its own learning-rate range
                trial["lr"] = float(f"{adam_lr * SGD_LR:.3g}") if value == "sgd" else adam_lr
            flags = f"{base} --label_fraction 0.1 --max_rounds 50 --patience 3 --tune " + " ".join(f"--{k} {v}" for k, v in trial.items())
            results = train(flags)
            rows.append({name: value, **{k: results.get(f"val_{k}") for k in ("accuracy", "top5_accuracy", "calibration_error", "loss")}})
        table = pd.DataFrame(rows)
        choice[name] = VALUES[name][int(table["accuracy"].idxmax())]  # the value itself, not a numpy number
        if name == "optimizer":
            choice["lr"] = float(f"{adam_lr * SGD_LR:.3g}") if choice[name] == "sgd" else adam_lr
        print(f"{setting}, {name}: chose {choice[name]}")
        display(table.round(4))
        fig, ax = plt.subplots(1, 2, figsize=(9, 3))
        for axis, metric in zip(ax, ("accuracy", "calibration_error")):
            axis.plot([str(v) for v in table[name]], table[metric], marker="o")
            axis.set_xlabel(name)
            axis.set_title(f"val {metric}")
        fig.suptitle(setting)
        plt.show()
    choices = json.load(open(CHOICE_FILE)) if os.path.exists(CHOICE_FILE) else {}
    choices[setting] = choice  # only now: the file holds finished tunings only
    with open(CHOICE_FILE, "w") as f:
        json.dump(choices, f, indent=1)
    return choice

## CNN-3
The final CNN-3 DAE first (`colab_final.ipynb` then skips it). Then fine-tuning, all three grids.

In [ ]:
!python train.py --mode pretrain --output_dir "{OUT}"

In [ ]:
tune("cnn3_finetune", "--mode finetune --encoder dae", 1e-4, GRID_1 + GRID_2 + GRID_3)

## ResNet-18
The ResNet-18 DAE first, as in `colab_final.ipynb` (30 epochs; about 2.5 hours). Then the linear probe, all three grids except the head's learning rate: only the head trains.

In [ ]:
!python train.py --mode pretrain {R} --num_epochs 30 --output_dir "{OUT}"

In [ ]:
tune("resnet18_linear_probe", f"--mode linear_probe --encoder dae {R}", 1e-3, [h for h in GRID_1 + GRID_2 + GRID_3 if h != "head_lr_mult"])

## The choices
What every setting chose. `colab_final.ipynb` uses them.

In [ ]:
pd.DataFrame(json.load(open(CHOICE_FILE))).T